# 07 — AdaBoost baseline

This notebook evaluates one simple, untuned AdaBoost model on the same fixed five-fold splits used by the other baselines

AdaBoost combines weak decision trees sequentially, giving more attention to observations misclassified by earlier trees. It is CPU-friendly at this dataset size

AdaBoost treats `sii` as a multiclass target and does not use the ordinal distance between classes during training. Quadratic Weighted Kappa is still used for evaluation

## Imports and data loading

In [1]:
import sys
from pathlib import Path

import numpy as np
import pandas as pd
from sklearn.ensemble import AdaBoostClassifier
from sklearn.pipeline import Pipeline
from sklearn.tree import DecisionTreeClassifier

# Make src importable when the notebook is launched from notebooks/
project_root = Path.cwd()
if project_root.name == "notebooks":
    project_root = project_root.parent
if str(project_root) not in sys.path:
    sys.path.insert(0, str(project_root))

from src.config import (
    ID_COLUMN,
    PROCESSED_DIR,
    RANDOM_STATE,
    RESULTS_DIR,
    TARGET,
    TRAIN_PATH,
)
from src.evaluation import create_cv_splits, evaluate_model
from src.features import build_features
from src.imputation import make_preprocessor

In [2]:
processed_train_path = PROCESSED_DIR / "train_features.parquet"

if processed_train_path.exists():
    train_features = pd.read_parquet(processed_train_path)
    print("Loaded Layer A features:", processed_train_path)
else:
    # Apply the same deterministic cleaning when notebook 06 has not been run yet
    train = pd.read_csv(TRAIN_PATH)
    train_features = build_features(train)
    print("Built Layer A features from:", TRAIN_PATH)

print("Feature table shape:", train_features.shape)

Loaded Layer A features: c:\Users\Honor\predict-internet-usage-ivanov-secret\data\processed\train_features.parquet
Feature table shape: (3960, 67)


## Dataset and fixed cross-validation splits

Only rows with a known target are used. `id` and `sii` are excluded from the model features. PCIAT leakage columns were already removed by Layer A

`create_cv_splits` reproduces the same stratified folds used by the baseline notebook, including stable fold membership after row reordering

In [3]:
labeled_train = (
    train_features[train_features[TARGET].notna()]
    .reset_index(drop=True)
)

feature_columns = [
    column
    for column in labeled_train.columns
    if column not in {ID_COLUMN, TARGET}
]

X = labeled_train[feature_columns].copy()
y = labeled_train[TARGET].astype(int)
ids = labeled_train[ID_COLUMN]

cv_splits = create_cv_splits(X=X, y=y, ids=ids)

print("X shape:", X.shape)
print("Target distribution:")
print(y.value_counts().sort_index())

X shape: (2736, 65)
Target distribution:
sii
0    1594
1     730
2     378
3      34
Name: count, dtype: int64


## AdaBoost model

A depth-1 decision tree is a deliberately weak learner, also called a decision stump. AdaBoost builds these trees sequentially and combines their votes

Median imputation and one-hot encoding are fit separately inside every training fold. Scaling is disabled because tree splits are unaffected by feature scale

In [4]:
adaboost_model = Pipeline(
    steps=[
        (
            "preprocessor",
            make_preprocessor(
                X,
                strategy="median",
                scale=False,  # Tree splits do not depend on feature scale
            ),
        ),
        (
            "model",
            AdaBoostClassifier(
                estimator=DecisionTreeClassifier(
                    max_depth=1,  # Keep each learner intentionally weak
                    random_state=RANDOM_STATE,
                ),
                n_estimators=50,  # Start with a small CPU-friendly ensemble
                learning_rate=1.0,  # Use sklearn's default update strength
                random_state=RANDOM_STATE,  # Reproduce estimator seeds
            ),
        ),
    ]
)

## Cross-validation evaluation

The shared evaluator reports training and validation QWK for every fold. A large gap between them indicates overfitting

In [5]:
adaboost_result = evaluate_model(
    model=adaboost_model,
    X=X,
    y=y,
    cv_splits=cv_splits,
)

Fold 1: training QWK=0.2600, validation QWK=0.2640
Fold 2: training QWK=0.3268, validation QWK=0.2798
Fold 3: training QWK=0.3665, validation QWK=0.3883
Fold 4: training QWK=0.3746, validation QWK=0.3475
Fold 5: training QWK=0.3144, validation QWK=0.2824
Mean training QWK: 0.3285
Mean validation QWK: 0.3124
Validation QWK standard deviation: 0.0475


## Save and compare results

The out-of-fold prediction counts show whether AdaBoost predicts every severity class or mostly follows the majority classes

In [6]:
training_scores = adaboost_result["training_scores"]
validation_scores = adaboost_result["validation_scores"]
oof_counts = np.bincount(
    adaboost_result["oof_predictions"],
    minlength=4,  # Include target classes with zero predictions
)

adaboost_results = pd.DataFrame(
    [
        {
            "model": "AdaBoostClassifier",
            **{
                f"fold_{fold_number}_qwk": score
                for fold_number, score in enumerate(
                    validation_scores,
                    start=1,
                )
            },
            "mean_training_qwk": np.mean(training_scores),
            "mean_validation_qwk": np.mean(validation_scores),
            "validation_std_qwk": np.std(validation_scores),
            **{
                f"oof_pred_{target_class}_count": count
                for target_class, count in enumerate(oof_counts)
            },
        }
    ]
).round(4)

RESULTS_DIR.mkdir(exist_ok=True)
results_path = RESULTS_DIR / "adaboost_cv_results.csv"
adaboost_results.to_csv(
    results_path,
    index=False,  # Keep the pandas row index out of the CSV
)

print("Saved results to:", results_path)
adaboost_results

Saved results to: c:\Users\Honor\predict-internet-usage-ivanov-secret\results\adaboost_cv_results.csv


,model,fold_1_qwk,fold_2_qwk,fold_3_qwk,fold_4_qwk,fold_5_qwk,mean_training_qwk,mean_validation_qwk,validation_std_qwk,oof_pred_0_count,oof_pred_1_count,oof_pred_2_count,oof_pred_3_count
0,AdaBoostClassifier,0.264,0.2798,0.3883,0.3475,0.2824,0.3285,0.3124,0.0475,2054,486,196,0


In [7]:
baseline_results_path = RESULTS_DIR / "baseline_cv_results.csv"
baseline_results = pd.read_csv(baseline_results_path)

# baseline_cv_results.csv is fit on raw, uncleaned features,
# uncomparable to AdaBoost, which runs on Layer A (build_features).
# The fair reference point is notebook 06's Ridge run on the same Layer A
# features AdaBoost uses here (0.3698 vs. raw baseline's 0.3677).
layer_a_ridge = (
    pd.read_csv(RESULTS_DIR / "imputation_cv_results.csv")
    .iloc[[0]]  # "Ridge | Layer A + median"
    .rename(columns={
        "setup": "model",
        "mean_val_qwk": "mean_validation_qwk",
        "val_std_qwk": "validation_std_qwk",
        "mean_train_qwk": "mean_training_qwk",
    })
)
layer_a_ridge["model"] = "Ridge (Layer A + median, notebook 06)"

comparison_columns = [
    "model",
    "mean_training_qwk",
    "mean_validation_qwk",
    "validation_std_qwk",
]

baseline_results_labeled = baseline_results.copy()
baseline_results_labeled["model"] = baseline_results_labeled["model"] + " (raw features, not directly comparable)"

comparison = pd.concat(
    [
        layer_a_ridge[comparison_columns],
        adaboost_results[comparison_columns],
        baseline_results_labeled[comparison_columns],
    ],
    ignore_index=True,
).sort_values("mean_validation_qwk", ascending=False)

comparison

,model,mean_training_qwk,mean_validation_qwk,validation_std_qwk
0,"Ridge (Layer A + median, notebook 06)",0.3992,0.3698,0.0280
3,"Ridge (raw features, not directly comparable)",0.4041,0.3677,0.0386
1,AdaBoostClassifier,0.3285,0.3124,0.0475
5,"RandomForestClassifier (raw features, not dire...",1.0000,0.2921,0.0249
4,"DecisionTreeClassifier (raw features, not dire...",1.0000,0.2209,0.0494
2,"DummyClassifier (raw features, not directly co...",0.0000,0.0000,0.0000


## AdaBoost experiments

The following experiments change one aspect at a time and keep the data, preprocessing, metric, and CV splits fixed

In [8]:
def make_adaboost_model(
    max_depth,
    n_estimators,
    learning_rate,
    class_weight=None,
):
    return Pipeline(
        steps=[
            (
                "preprocessor",
                make_preprocessor(
                    X,
                    strategy="median",
                    scale=False,  # Tree splits do not depend on feature scale
                ),
            ),
            (
                "model",
                AdaBoostClassifier(
                    estimator=DecisionTreeClassifier(
                        max_depth=max_depth,
                        class_weight=class_weight,
                        random_state=RANDOM_STATE,
                    ),
                    n_estimators=n_estimators,
                    learning_rate=learning_rate,
                    random_state=RANDOM_STATE,
                ),
            ),
        ]
    )


def result_row(name, result, max_depth, n_estimators, learning_rate, class_weight):
    training_scores = result["training_scores"]
    validation_scores = result["validation_scores"]
    oof_counts = np.bincount(
        result["oof_predictions"],
        minlength=4,  # Include target classes with zero predictions
    )

    return {
        "experiment": name,
        "max_depth": max_depth,
        "n_estimators": n_estimators,
        "learning_rate": learning_rate,
        "class_weight": class_weight or "none",
        "mean_training_qwk": np.mean(training_scores),
        "mean_validation_qwk": np.mean(validation_scores),
        "validation_std_qwk": np.std(validation_scores),
        "train_validation_gap": (
            np.mean(training_scores) - np.mean(validation_scores)
        ),
        **{
            f"oof_pred_{target_class}_count": count
            for target_class, count in enumerate(oof_counts)
        },
    }

### Experiment 1: tree depth

Increasing `max_depth` lets each weak learner model more complex relationships. Depths 1, 2, and 3 are compared while the ensemble size and learning rate stay fixed

In [9]:
depth_results = {1: adaboost_result}

for max_depth in [2, 3]:
    print(f"\nAdaBoost | max_depth={max_depth}")
    model = make_adaboost_model(
        max_depth=max_depth,
        n_estimators=50,
        learning_rate=1.0,
    )
    depth_results[max_depth] = evaluate_model(
        model=model,
        X=X,
        y=y,
        cv_splits=cv_splits,
    )

depth_table = pd.DataFrame(
    [
        result_row(
            name=f"depth_{max_depth}",
            result=result,
            max_depth=max_depth,
            n_estimators=50,
            learning_rate=1.0,
            class_weight=None,
        )
        for max_depth, result in depth_results.items()
    ]
).round(4)

depth_table[
    [
        "max_depth",
        "mean_training_qwk",
        "mean_validation_qwk",
        "validation_std_qwk",
        "train_validation_gap",
    ]
]


AdaBoost | max_depth=2
Fold 1: training QWK=0.4182, validation QWK=0.3286
Fold 2: training QWK=0.3979, validation QWK=0.3739
Fold 3: training QWK=0.3922, validation QWK=0.3329
Fold 4: training QWK=0.4225, validation QWK=0.2929
Fold 5: training QWK=0.4456, validation QWK=0.3051
Mean training QWK: 0.4153
Mean validation QWK: 0.3267
Validation QWK standard deviation: 0.0279

AdaBoost | max_depth=3
Fold 1: training QWK=0.5013, validation QWK=0.2688
Fold 2: training QWK=0.5229, validation QWK=0.3403
Fold 3: training QWK=0.4530, validation QWK=0.3630
Fold 4: training QWK=0.4779, validation QWK=0.2162
Fold 5: training QWK=0.4785, validation QWK=0.2666
Mean training QWK: 0.4867
Mean validation QWK: 0.2910
Validation QWK standard deviation: 0.0535


,max_depth,mean_training_qwk,mean_validation_qwk,validation_std_qwk,train_validation_gap
0,1,0.3285,0.3124,0.0475,0.0160
1,2,0.4153,0.3267,0.0279,0.0886
2,3,0.4867,0.2910,0.0535,0.1957


**Conclusion:** depth 2 improves mean validation QWK from **0.3124** to **0.3257**. Depth 3 lowers it to **0.2936** and increases the train-validation gap, so depth 2 is selected

### Experiment 2: ensemble size and learning rate

More trees are paired with a smaller learning rate so each boosting step makes a more gradual correction. Tree depth remains fixed at 2

In [10]:
parameter_results = {
    (50, 1.0): depth_results[2],
}

for n_estimators, learning_rate in [(100, 0.5), (200, 0.2)]:
    print(
        f"\nAdaBoost | n_estimators={n_estimators}, "
        f"learning_rate={learning_rate}"
    )
    model = make_adaboost_model(
        max_depth=2,
        n_estimators=n_estimators,
        learning_rate=learning_rate,
    )
    parameter_results[(n_estimators, learning_rate)] = evaluate_model(
        model=model,
        X=X,
        y=y,
        cv_splits=cv_splits,
    )

parameter_table = pd.DataFrame(
    [
        result_row(
            name=f"estimators_{n_estimators}_rate_{learning_rate}",
            result=result,
            max_depth=2,
            n_estimators=n_estimators,
            learning_rate=learning_rate,
            class_weight=None,
        )
        for (n_estimators, learning_rate), result in parameter_results.items()
    ]
).round(4)

parameter_table[
    [
        "n_estimators",
        "learning_rate",
        "mean_training_qwk",
        "mean_validation_qwk",
        "validation_std_qwk",
        "train_validation_gap",
    ]
]


AdaBoost | n_estimators=100, learning_rate=0.5
Fold 1: training QWK=0.4017, validation QWK=0.3477
Fold 2: training QWK=0.3844, validation QWK=0.3569
Fold 3: training QWK=0.3811, validation QWK=0.3876
Fold 4: training QWK=0.4131, validation QWK=0.2741
Fold 5: training QWK=0.4275, validation QWK=0.3542
Mean training QWK: 0.4016
Mean validation QWK: 0.3441
Validation QWK standard deviation: 0.0376

AdaBoost | n_estimators=200, learning_rate=0.2
Fold 1: training QWK=0.3965, validation QWK=0.3956
Fold 2: training QWK=0.3603, validation QWK=0.3550
Fold 3: training QWK=0.3707, validation QWK=0.3496
Fold 4: training QWK=0.4115, validation QWK=0.3047
Fold 5: training QWK=0.4168, validation QWK=0.3358
Mean training QWK: 0.3912
Mean validation QWK: 0.3481
Validation QWK standard deviation: 0.0295


,n_estimators,learning_rate,mean_training_qwk,mean_validation_qwk,validation_std_qwk,train_validation_gap
0,50,1.0,0.4153,0.3267,0.0279,0.0886
1,100,0.5,0.4016,0.3441,0.0376,0.0575
2,200,0.2,0.3912,0.3481,0.0295,0.0430


**Conclusion:** 200 trees with `learning_rate=0.2` perform best, reaching mean validation QWK **0.3481**. The smaller learning rate also reduces the train-validation gap

**Design caveat:** the experiment changes `n_estimators` and `learning_rate` together in both tested combinations (100, 0.5) and (200, 0.2), so the gain can't be attributed to either parameter alone ("one aspect at a time" isn't true). This is fixed below (Task 3): Optuna samples all 4 hyperparameters independently over many trials, and `optuna.importance.get_param_importances` reports each one's individual contribution to the QWK variance.

### Experiment 3: balanced class weights

Balanced weights give larger importance to rare target classes. This may improve rare-class recall, but it can also produce too many severe-class predictions

In [11]:
balanced_model = make_adaboost_model(
    max_depth=2,
    n_estimators=200,
    learning_rate=0.2,
    class_weight="balanced",
)

balanced_result = evaluate_model(
    model=balanced_model,
    X=X,
    y=y,
    cv_splits=cv_splits,
)

balance_table = pd.DataFrame(
    [
        result_row(
            name="unbalanced",
            result=parameter_results[(200, 0.2)],
            max_depth=2,
            n_estimators=200,
            learning_rate=0.2,
            class_weight=None,
        ),
        result_row(
            name="balanced",
            result=balanced_result,
            max_depth=2,
            n_estimators=200,
            learning_rate=0.2,
            class_weight="balanced",
        ),
    ]
).round(4)

balance_table[
    [
        "class_weight",
        "mean_validation_qwk",
        "oof_pred_0_count",
        "oof_pred_1_count",
        "oof_pred_2_count",
        "oof_pred_3_count",
    ]
]

Fold 1: training QWK=0.3511, validation QWK=0.3168
Fold 2: training QWK=0.3318, validation QWK=0.3621
Fold 3: training QWK=0.3581, validation QWK=0.3608
Fold 4: training QWK=0.3337, validation QWK=0.2836
Fold 5: training QWK=0.3673, validation QWK=0.3161
Mean training QWK: 0.3484
Mean validation QWK: 0.3279
Validation QWK standard deviation: 0.0299


,class_weight,mean_validation_qwk,oof_pred_0_count,oof_pred_1_count,oof_pred_2_count,oof_pred_3_count
0,none,0.3481,2026,568,141,1
1,balanced,0.3279,1595,228,642,271


**Conclusion:** balanced weights lower mean validation QWK from **0.3481** to **0.3279**. They increase class 3 predictions from 1 to 271 although class 3 has only 34 training examples, so balanced weights are not selected

## Final experiment comparison

In [12]:
experiment_results = pd.concat(
    [
        depth_table,
        parameter_table.iloc[1:],  # Depth 2 with 50 trees is already included
        balance_table.iloc[[1]],  # The unbalanced setup is already included
    ],
    ignore_index=True,
).sort_values("mean_validation_qwk", ascending=False)

experiment_results_path = (
    RESULTS_DIR / "adaboost_experiment_results.csv"
)
experiment_results.to_csv(
    experiment_results_path,
    index=False,  # Keep the pandas row index out of the CSV
)

print("Saved results to:", experiment_results_path)
experiment_results[
    [
        "experiment",
        "max_depth",
        "n_estimators",
        "learning_rate",
        "class_weight",
        "mean_training_qwk",
        "mean_validation_qwk",
        "validation_std_qwk",
    ]
]

Saved results to: c:\Users\Honor\predict-internet-usage-ivanov-secret\results\adaboost_experiment_results.csv


,experiment,max_depth,n_estimators,learning_rate,class_weight,mean_training_qwk,mean_validation_qwk,validation_std_qwk
4,estimators_200_rate_0.2,2,200,0.2,none,0.3912,0.3481,0.0295
3,estimators_100_rate_0.5,2,100,0.5,none,0.4016,0.3441,0.0376
5,balanced,2,200,0.2,balanced,0.3484,0.3279,0.0299
1,depth_2,2,50,1.0,none,0.4153,0.3267,0.0279
0,depth_1,1,50,1.0,none,0.3285,0.3124,0.0475
2,depth_3,3,50,1.0,none,0.4867,0.2910,0.0535


## Final conclusion

The best AdaBoost setup uses depth 2, 200 estimators, `learning_rate=0.2`, and no class balancing. It improves mean validation QWK from **0.3124** to **0.3481**, but remains below Ridge on the same Layer A features at **0.3698**

These settings were selected using the same CV results shown here, so **0.3481 is an exploratory tuning score**, not a new unbiased performance estimate. The "Experiment 2" grid above also conflates `n_estimators` and `learning_rate`, so its contribution to that score is not individually attributable — see the design caveat below it, and Task 3's parameter-importance analysis for a proper breakdown.

## Task 1: comparing 3 ways to turn AdaBoost output into an ordinal score

The classifier above throws away the ordinal structure of `sii`: `argmax` over 4 classes treats 2 adjacent classes exactly like 2 opposite ones. 2 alternatives keep more of that structure:

1. **Classifier -> score**: take `predict_proba` from the same AdaBoostClassifier and use the probability-weighted class index (`E[class]`) as a continuous score, then round it to a class with optimized thresholds.
2. **AdaBoostRegressor**: fit a regressor directly on the numeric `sii` target and round its continuous output the same way.

All 3 variants share the *same* base configuration (`max_depth=1`, `n_estimators=50`, `learning_rate=1.0`, the notebook's original settings) so the comparison isolates the effect of the encoding, not the model.

Thresholds for variants 2 and 3 are fit the same way as in the XGBoost tuning notebook: on a holdout carved out of each outer *training* fold, never on the outer validation fold, so nothing about the outer validation fold leaks into the thresholds used to score it.

In [13]:
from scipy.optimize import minimize
from sklearn.ensemble import AdaBoostRegressor
from sklearn.metrics import cohen_kappa_score
from sklearn.model_selection import train_test_split
from sklearn.tree import DecisionTreeRegressor

from src.evaluation import quadratic_weighted_kappa

CLASS_VALUES = np.array([0, 1, 2, 3])


def predictions_to_classes(predictions, thresholds):
    return np.digitize(np.asarray(predictions), np.sort(thresholds))


def _negative_qwk(thresholds, predictions, y_true):
    return -cohen_kappa_score(y_true, predictions_to_classes(predictions, thresholds), weights="quadratic")


def optimize_thresholds(predictions, y_true, initial=(0.5, 1.5, 2.5)):
    result = minimize(
        _negative_qwk, np.asarray(initial, dtype=float),
        args=(np.asarray(predictions), np.asarray(y_true)), method="Nelder-Mead",
    )
    return np.sort(result.x)


def fit_predict_oof_scored(build_model, predict_score, X_frame, y, cv_splits, threshold_holdout_fraction=0.15, verbose=True):
    """Fold-safe OOF class predictions for any model producing a continuous ordinal score."""
    oof_classes = np.zeros(len(y), dtype=int)
    fold_scores = []
    fold_thresholds = []
    for fold, (train_idx, val_idx) in enumerate(cv_splits, start=1):
        X_outer, y_outer = X_frame.iloc[train_idx], y.iloc[train_idx]
        fit_pos, threshold_pos = train_test_split(
            np.arange(len(X_outer)), test_size=threshold_holdout_fraction,
            random_state=RANDOM_STATE, stratify=y_outer,
        )
        model = build_model()
        model.fit(X_outer.iloc[fit_pos], y_outer.iloc[fit_pos])

        threshold_scores = predict_score(model, X_outer.iloc[threshold_pos])
        thresholds = optimize_thresholds(threshold_scores, y_outer.iloc[threshold_pos])
        fold_thresholds.append(thresholds)

        val_scores = predict_score(model, X_frame.iloc[val_idx])
        oof_classes[val_idx] = predictions_to_classes(val_scores, thresholds)

        fold_score = quadratic_weighted_kappa(y.iloc[val_idx], oof_classes[val_idx])
        fold_scores.append(fold_score)
        if verbose:
            print(f"Fold {fold}: validation QWK={fold_score:.4f} (thresholds={thresholds.round(3)})")
    return oof_classes, fold_scores, fold_thresholds

### Variant 2: classifier probability -> score

In [14]:
def build_original_classifier_pipeline():
    return Pipeline([
        ("preprocessor", make_preprocessor(X, strategy="median", scale=False)),
        ("model", AdaBoostClassifier(
            estimator=DecisionTreeClassifier(max_depth=1, random_state=RANDOM_STATE),
            n_estimators=50, learning_rate=1.0, random_state=RANDOM_STATE,
        )),
    ])


def classifier_expected_score(model, X_slice):
    return model.predict_proba(X_slice) @ CLASS_VALUES


classifier_score_oof, classifier_score_fold_scores, classifier_score_thresholds = fit_predict_oof_scored(
    build_original_classifier_pipeline, classifier_expected_score, X, y, cv_splits,
)
classifier_score_qwk = quadratic_weighted_kappa(y, classifier_score_oof)
print("\nClassifier probability-score QWK:", round(classifier_score_qwk, 4))

Fold 1: validation QWK=0.0000 (thresholds=[0.5 1.5 2.5])
Fold 2: validation QWK=0.1970 (thresholds=[0.51  1.466 2.491])
Fold 3: validation QWK=0.0133 (thresholds=[0.5 1.5 2.5])
Fold 4: validation QWK=0.1752 (thresholds=[0.515 1.473 2.532])
Fold 5: validation QWK=0.1573 (thresholds=[0.514 1.474 2.467])

Classifier probability-score QWK: 0.1229


### Variant 3: AdaBoostRegressor

In [15]:
def build_original_regressor_pipeline():
    return Pipeline([
        ("preprocessor", make_preprocessor(X, strategy="median", scale=False)),
        ("model", AdaBoostRegressor(
            estimator=DecisionTreeRegressor(max_depth=1, random_state=RANDOM_STATE),
            n_estimators=50, learning_rate=1.0, random_state=RANDOM_STATE,
        )),
    ])


def regressor_score(model, X_slice):
    return model.predict(X_slice)


regressor_oof, regressor_fold_scores, regressor_thresholds = fit_predict_oof_scored(
    build_original_regressor_pipeline, regressor_score, X, y, cv_splits,
)
regressor_qwk = quadratic_weighted_kappa(y, regressor_oof)
print("\nAdaBoostRegressor QWK:", round(regressor_qwk, 4))

Fold 1: validation QWK=0.3118 (thresholds=[0.5 1.5 2.5])
Fold 2: validation QWK=0.0000 (thresholds=[0.5 1.5 2.5])
Fold 3: validation QWK=0.2846 (thresholds=[0.5 1.5 2.5])
Fold 4: validation QWK=0.0000 (thresholds=[0.5 1.5 2.5])
Fold 5: validation QWK=0.2508 (thresholds=[0.5 1.5 2.5])

AdaBoostRegressor QWK: 0.174


In [16]:
encoding_comparison = pd.DataFrame([
    {
        "variant": "AdaBoostClassifier (argmax)",
        "mean_val_qwk": np.mean(adaboost_result["validation_scores"]),
        "val_std_qwk": np.std(adaboost_result["validation_scores"]),
    },
    {
        "variant": "AdaBoostClassifier (proba -> score, optimized thresholds)",
        "mean_val_qwk": classifier_score_qwk,
        "val_std_qwk": np.std(classifier_score_fold_scores),
    },
    {
        "variant": "AdaBoostRegressor (optimized thresholds)",
        "mean_val_qwk": regressor_qwk,
        "val_std_qwk": np.std(regressor_fold_scores),
    },
]).round(4).sort_values("mean_val_qwk", ascending=False)

encoding_comparison_path = RESULTS_DIR / "adaboost_encoding_comparison.csv"
encoding_comparison.to_csv(encoding_comparison_path, index=False)
print("Saved results to:", encoding_comparison_path)
encoding_comparison

Saved results to: c:\Users\Honor\predict-internet-usage-ivanov-secret\results\adaboost_encoding_comparison.csv


,variant,mean_val_qwk,val_std_qwk
0,AdaBoostClassifier (argmax),0.3124,0.0475
2,AdaBoostRegressor (optimized thresholds),0.1740,0.1397
1,"AdaBoostClassifier (proba -> score, optimized ...",0.1229,0.0843


**Conclusion:** at the shared baseline configuration (`max_depth=1`, 50 estimators, `learning_rate=1.0`), the plain classifier wins clearly — argmax **0.3124** vs. probability-score **0.1229** vs. regressor **0.1740**. Both continuous-score variants are also far less stable across folds (std 0.08–0.14 vs. 0.05 for the classifier); the regressor even collapses to predicting a single class for the entire validation fold in 2 of 5 folds (see the Task 2 note below).

This is a property of *this specific weak baseline*, not a general statement that regression is worse than classification for an ordinal target, Task 3 tunes the regressor and reaches **0.4159**, well above the classifier's best grid-search result (0.3481, see the "AdaBoost experiments" section above). At `max_depth=1` the model's continuous output range is simply too narrow for thresholding to add anything over a hard argmax.

## Task 2: imputation strategies for AdaBoostRegressor

3 preprocessing variants for the same fixed AdaBoostRegressor configuration (`max_depth=1`, `n_estimators=50`, `learning_rate=1.0`, matching the baseline used in Task 2):

1. **Median** — `make_preprocessor(..., strategy="median")`, the default used everywhere above.
2. **Median + missing indicators** — the same median imputation, plus 1 binary "was this cell missing" column per numeric feature (`SimpleImputer(add_indicator=True)`).
3. **Iterative** — the existing `strategy="iterative"` option (`IterativeImputer` for the linked BIA/Physical blocks, median elsewhere).

In [17]:
from sklearn.compose import ColumnTransformer
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import OneHotEncoder


def make_median_indicator_preprocessor(feature_frame):
    """Median imputation plus one missing-value indicator per numeric column."""
    categorical_columns = feature_frame.select_dtypes(include=["object", "string", "category"]).columns.tolist()
    numeric_columns = feature_frame.select_dtypes(include="number").columns.tolist()
    return ColumnTransformer(
        transformers=[
            ("numeric", SimpleImputer(strategy="median", add_indicator=True), numeric_columns),
            (
                "categorical",
                Pipeline([
                    ("imputer", SimpleImputer(strategy="most_frequent")),
                    ("encoder", OneHotEncoder(handle_unknown="ignore")),
                ]),
                categorical_columns,
            ),
        ],
        remainder="drop",
    )


def build_regressor_pipeline(preprocessor_factory):
    return Pipeline([
        ("preprocessor", preprocessor_factory(X)),
        ("model", AdaBoostRegressor(
            estimator=DecisionTreeRegressor(max_depth=1, random_state=RANDOM_STATE),
            n_estimators=50, learning_rate=1.0, random_state=RANDOM_STATE,
        )),
    ])


imputation_variants = {
    "median": lambda frame: make_preprocessor(frame, strategy="median", scale=False),
    "median_with_indicators": make_median_indicator_preprocessor,
    "iterative": lambda frame: make_preprocessor(frame, strategy="iterative", scale=False),
}

imputation_rows = []
for name, preprocessor_factory in imputation_variants.items():
    print(f"\nAdaBoostRegressor | imputation={name}")
    oof_classes, fold_scores, _ = fit_predict_oof_scored(
        lambda pf=preprocessor_factory: build_regressor_pipeline(pf),
        regressor_score, X, y, cv_splits,
    )
    imputation_rows.append({
        "imputation": name,
        "mean_val_qwk": quadratic_weighted_kappa(y, oof_classes),
        "val_std_qwk": np.std(fold_scores),
    })

imputation_table = pd.DataFrame(imputation_rows).round(4).sort_values("mean_val_qwk", ascending=False)
imputation_results_path = RESULTS_DIR / "adaboost_imputation_comparison.csv"
imputation_table.to_csv(imputation_results_path, index=False)
print("\nSaved results to:", imputation_results_path)
imputation_table


AdaBoostRegressor | imputation=median
Fold 1: validation QWK=0.3118 (thresholds=[0.5 1.5 2.5])
Fold 2: validation QWK=0.0000 (thresholds=[0.5 1.5 2.5])
Fold 3: validation QWK=0.2846 (thresholds=[0.5 1.5 2.5])
Fold 4: validation QWK=0.0000 (thresholds=[0.5 1.5 2.5])
Fold 5: validation QWK=0.2508 (thresholds=[0.5 1.5 2.5])

AdaBoostRegressor | imputation=median_with_indicators
Fold 1: validation QWK=0.3118 (thresholds=[0.5 1.5 2.5])
Fold 2: validation QWK=0.0000 (thresholds=[0.5 1.5 2.5])
Fold 3: validation QWK=0.2846 (thresholds=[0.5 1.5 2.5])
Fold 4: validation QWK=0.0000 (thresholds=[0.5 1.5 2.5])
Fold 5: validation QWK=0.2508 (thresholds=[0.5 1.5 2.5])

AdaBoostRegressor | imputation=iterative


c:\Users\Honor\AppData\Local\Programs\Python\Python313\Lib\site-packages\sklearn\impute\_iterative.py:867: ConvergenceWarning: [IterativeImputer] Early stopping criterion not reached.
  warnings.warn(


Fold 1: validation QWK=0.3118 (thresholds=[0.5 1.5 2.5])
Fold 2: validation QWK=0.0000 (thresholds=[0.5 1.5 2.5])
Fold 3: validation QWK=0.2846 (thresholds=[0.5 1.5 2.5])


c:\Users\Honor\AppData\Local\Programs\Python\Python313\Lib\site-packages\sklearn\impute\_iterative.py:867: ConvergenceWarning: [IterativeImputer] Early stopping criterion not reached.
  warnings.warn(


Fold 4: validation QWK=0.0000 (thresholds=[0.5 1.5 2.5])
Fold 5: validation QWK=0.2508 (thresholds=[0.5 1.5 2.5])

Saved results to: c:\Users\Honor\predict-internet-usage-ivanov-secret\results\adaboost_imputation_comparison.csv


,imputation,mean_val_qwk,val_std_qwk
0,median,0.174,0.1397
1,median_with_indicators,0.174,0.1397
2,iterative,0.174,0.1397


**Conclusion:** all three imputation strategies score **exactly the same** — mean QWK **0.1740**, std **0.1397**, identical fold-by-fold.

The reason the difference never survives to QWK is that this baseline (`max_depth=1`, 50 estimators) is too weak to separate the classes: its predicted scores land in a narrow band (e.g. **0.60–1.29** in one fold), so almost every prediction rounds to class 1 regardless of the small shifts imputation causes — in fold 2, for example, the model predicts class 1 for **all 547** validation rows, which forces QWK to 0 no matter which imputation was used. A coarse, saturated model simply can't reveal whether imputation quality matters.

**Take-away:** imputation choice should be re-compared once the regressor is actually expressive enough to use its input (see Task 3) — this comparison, as scoped (fixed weak config), correctly reports "no measurable difference," but that is a statement about this baseline's ceiling.

## Task 3 — hyperparameter tuning for AdaBoostRegressor

Optuna tunes the base tree's `max_depth` and `min_samples_leaf`, plus the ensemble's `n_estimators` and `learning_rate`, using the imputation strategy that won Task 2. The objective reuses `fit_predict_oof_scored`, so thresholds stay fold-local inside every trial — the same protocol used everywhere else in this notebook and in the XGBoost tuning notebook. The result is compared against the untuned regressor from Task 1 / Task 2.

Unlike the manual "Experiment 2" grid above, Optuna varies all four hyperparameters independently across 50 trials, and `optuna.importance.get_param_importances` (fANOVA) below reports each parameter's individual share of the QWK variance — the proper way to isolate each parameter's contribution instead of changing two at once.

In [18]:
import optuna
optuna.logging.set_verbosity(optuna.logging.WARNING)

best_imputation_name = imputation_table.iloc[0]["imputation"]
best_imputation_factory = imputation_variants[best_imputation_name]
print("Tuning with imputation strategy:", best_imputation_name)


def objective(trial):
    params = {
        "max_depth": trial.suggest_int("max_depth", 1, 4),
        "min_samples_leaf": trial.suggest_int("min_samples_leaf", 1, 30),
        "n_estimators": trial.suggest_int("n_estimators", 25, 300),
        "learning_rate": trial.suggest_float("learning_rate", 0.05, 1.0, log=True),
    }

    def build():
        return Pipeline([
            ("preprocessor", best_imputation_factory(X)),
            ("model", AdaBoostRegressor(
                estimator=DecisionTreeRegressor(
                    max_depth=params["max_depth"],
                    min_samples_leaf=params["min_samples_leaf"],
                    random_state=RANDOM_STATE,
                ),
                n_estimators=params["n_estimators"],
                learning_rate=params["learning_rate"],
                random_state=RANDOM_STATE,
            )),
        ])

    oof_classes, _, _ = fit_predict_oof_scored(build, regressor_score, X, y, cv_splits, verbose=False)
    return quadratic_weighted_kappa(y, oof_classes)


study = optuna.create_study(direction="maximize", sampler=optuna.samplers.TPESampler(seed=RANDOM_STATE))
study.optimize(objective, n_trials=50, show_progress_bar=True)
print("Best tuning QWK:", round(study.best_value, 4))
print("Best params:", study.best_params)

c:\Users\Honor\AppData\Local\Programs\Python\Python313\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


Tuning with imputation strategy: median


Best trial: 14. Best value: 0.415943: 100%|██████████| 50/50 [02:18<00:00,  2.76s/it]

Best tuning QWK: 0.4159
Best params: {'max_depth': 4, 'min_samples_leaf': 8, 'n_estimators': 43, 'learning_rate': 0.16635453196840702}


In [19]:
param_importances = optuna.importance.get_param_importances(study)
importance_table = pd.DataFrame(
    [{"parameter": name, "importance": value} for name, value in param_importances.items()]
).sort_values("importance", ascending=False)

importance_path = RESULTS_DIR / "adaboost_regressor_param_importance.csv"
importance_table.to_csv(importance_path, index=False)
print("Saved results to:", importance_path)
print("\nParameter importance (fANOVA — each parameter's individual share of the QWK variance):")
importance_table

Saved results to: c:\Users\Honor\predict-internet-usage-ivanov-secret\results\adaboost_regressor_param_importance.csv

Parameter importance (fANOVA — each parameter's individual share of the QWK variance):


,parameter,importance
0,learning_rate,0.741386
1,max_depth,0.183434
2,n_estimators,0.044835
3,min_samples_leaf,0.030345


In [ ]:
def build_tuned_regressor_pipeline():
    return Pipeline([
        ("preprocessor", best_imputation_factory(X)),
        ("model", AdaBoostRegressor(
            estimator=DecisionTreeRegressor(
                max_depth=study.best_params["max_depth"],
                min_samples_leaf=study.best_params["min_samples_leaf"],
                random_state=RANDOM_STATE,
            ),
            n_estimators=study.best_params["n_estimators"],
            learning_rate=study.best_params["learning_rate"],
            random_state=RANDOM_STATE,
        )),
    ])


tuned_oof, tuned_fold_scores, tuned_thresholds = fit_predict_oof_scored(
    build_tuned_regressor_pipeline, regressor_score, X, y, cv_splits,
)
tuned_qwk = quadratic_weighted_kappa(y, tuned_oof)
print("\nTuned AdaBoostRegressor QWK:", round(tuned_qwk, 4))

imputation_indexed = imputation_table.set_index("imputation")
regressor_tuning_comparison = pd.DataFrame([
    {
        "setup": "AdaBoostRegressor | baseline (depth=1, 50 estimators, rate=1.0, median imputation)",
        "mean_val_qwk": regressor_qwk,
        "val_std_qwk": np.std(regressor_fold_scores),
    },
    {
        "setup": f"AdaBoostRegressor | best Task 2 imputation ({best_imputation_name}), untuned",
        "mean_val_qwk": imputation_indexed.loc[best_imputation_name, "mean_val_qwk"],
        "val_std_qwk": imputation_indexed.loc[best_imputation_name, "val_std_qwk"],
    },
    {
        "setup": f"AdaBoostRegressor | Optuna-tuned ({best_imputation_name} imputation)",
        "mean_val_qwk": tuned_qwk,
        "val_std_qwk": np.std(tuned_fold_scores),
    },
]).round(4).sort_values("mean_val_qwk", ascending=False)

regressor_tuning_path = RESULTS_DIR / "adaboost_regressor_tuning.csv"
regressor_tuning_comparison.to_csv(regressor_tuning_path, index=False)
print("Saved results to:", regressor_tuning_path)
regressor_tuning_comparison

**Conclusion:** tuning transforms the regressor — mean QWK goes from **0.1740** (baseline) to **0.4159** (Optuna, 50 trials, same best trial found already at 30), confirming that Task 2's "no difference between imputations" result was purely a ceiling effect of the deliberately weak `max_depth=1` baseline. The winning configuration is `max_depth=4`, `min_samples_leaf=8`, `n_estimators=43`, `learning_rate≈0.166`, with `median` imputation (Task 2's imputation strategies were tied, so the median default was kept).

**Parameter importance (fANOVA)** answers the "which parameter actually drove the gain" question that Experiment 2's combined grid couldn't: `learning_rate` explains **74%** of the QWK variance across trials, `max_depth` **18%**, `n_estimators` **4%**, `min_samples_leaf` **3%**. So the individual driver behind AdaBoost's gains is overwhelmingly the learning rate — not the ensemble size that Experiment 2 varied jointly with it.

At **0.4159**, the tuned AdaBoostRegressor is now competitive with the classifier grid-search's best result (0.3481) and close to the honest, fold-safe XGBoost result from `08_xgboost_tuning.ipynb` (**0.4145**), and above Ridge on the same Layer A features (**0.3698**). These trials were scored on the same CV splits reported here, so **0.4159 is an exploratory tuning score**, not an unbiased estimate, but the hyperparameter search itself still reuses the reporting splits.